# Module 7 - Causal Inference: Propensity Score Methods

**Goal:** learn the three standard tools for adjusting observational data so a treatment-vs-outcome comparison approaches the estimate you would get from a randomized trial.

Randomization balances confounders by design. Observational data (FAERS, Medicare, EHRs, chart reviews) does not. If older patients are both more likely to receive a treatment AND more likely to have the outcome, a naive comparison confuses the treatment effect with the age effect. Propensity-score methods try to fix that.

**What we will cover:**
1. Simulated data with a KNOWN confounder, so we can see the naive analysis lie and confirm the correction works.
2. **Propensity Score Matching (PSM)**: pair each treated unit with the most similar control.
3. **Inverse Probability of Treatment Weighting (IPTW)**: reweight the whole sample so the covariate distributions match.
4. **Balance diagnostics**: standardized mean differences and propensity overlap plots.
5. Apply to the NCCTG colon trial (real data from the SEER module) and see what the propensity-adjusted mortality contrast looks like.
6. **E-value**: how strong would an unmeasured confounder have to be to explain the finding away?

**Why this belongs in the portfolio:** every RWE / HEOR / pharmacoepi role tests for these skills. FAERS specifically cannot support them (no covariate depth), but the framework transfers cleanly to Medicare / SEER / EHR work.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)  # reproducibility for the simulation

## 1. Simulated data with a known confounder

We generate a dataset where:
- **Confounder**: `age` (older patients are both more likely to be treated AND more likely to have the outcome).
- **True treatment effect** on outcome probability: `-0.15` (protective, 15 percentage-point absolute risk reduction).
- **Age effect** on outcome: `+0.005 per year over 60`.

Because we know the ground truth, we can score every method against it.

In [ ]:
n = 2000
age = rng.normal(65, 10, n).clip(40, 90)

# Treatment probability is a logistic function of age (older -> more likely treated).
propensity_true = 1 / (1 + np.exp(-(age - 65) / 5))
treatment = rng.binomial(1, propensity_true)

# Outcome probability = baseline + true treatment effect + age effect.
TRUE_TE = -0.15
outcome_prob = (0.35 + TRUE_TE * treatment + 0.005 * (age - 60)).clip(0.01, 0.99)
outcome = rng.binomial(1, outcome_prob)

sim = pd.DataFrame({'age': age, 'treatment': treatment, 'outcome': outcome})
print(f'Total n = {len(sim)}')
print(f'Treated: {sim.treatment.sum():,}  |  Control: {(sim.treatment==0).sum():,}')
print(f'Mean age treated : {sim[sim.treatment==1].age.mean():.1f}')
print(f'Mean age control : {sim[sim.treatment==0].age.mean():.1f}')
print(f'\nTRUE treatment effect on outcome probability: {TRUE_TE:+.3f}')

### Naive analysis: unadjusted risk difference

Just take the raw difference in outcome rate between treated and control groups.

In [ ]:
p_treated = sim[sim.treatment == 1].outcome.mean()
p_control = sim[sim.treatment == 0].outcome.mean()
naive_rd = p_treated - p_control

print(f'P(outcome | treated) = {p_treated:.3f}')
print(f'P(outcome | control) = {p_control:.3f}')
print(f'Naive risk difference = {naive_rd:+.3f}')
print(f'True treatment effect = {TRUE_TE:+.3f}')
print(f'\nBias in naive estimate = {naive_rd - TRUE_TE:+.3f}')
print('The naive estimate is biased UPWARD because older patients')
print('are overrepresented in the treated group AND more likely to have the outcome.')

## 2. Propensity Score Matching (PSM)

**Idea:** For every treated unit, find the control unit whose *propensity score* (predicted probability of being treated) is closest. Compare the matched pairs.

The propensity score is a single number that summarizes all covariates. Matching on it is (asymptotically) equivalent to matching on the covariates themselves.

In [ ]:
# 1. Fit the propensity model (any classifier will do; logistic is standard).
ps_model = LogisticRegression()
ps_model.fit(sim[['age']], sim['treatment'])
sim['ps'] = ps_model.predict_proba(sim[['age']])[:, 1]

# 2. 1:1 nearest-neighbor matching (with replacement for simplicity).
treated = sim[sim.treatment == 1].copy().reset_index(drop=True)
control = sim[sim.treatment == 0].copy().reset_index(drop=True)

tree = cKDTree(control[['ps']].values)
_, match_idx = tree.query(treated[['ps']].values, k=1)
control_matched = control.iloc[match_idx].reset_index(drop=True)

# 3. Compare matched groups.
psm_rd = treated.outcome.mean() - control_matched.outcome.mean()
print(f'Matched sample size: {len(treated):,} treated + {len(control_matched):,} matched controls')
print(f'PSM risk difference = {psm_rd:+.3f}')
print(f'True treatment effect = {TRUE_TE:+.3f}')
print(f'Bias after PSM = {psm_rd - TRUE_TE:+.3f}')

## 3. Inverse Probability of Treatment Weighting (IPTW)

**Idea:** Rather than matching, use every unit but *reweight* it. Treated units get weight `1/ps`, control units get weight `1/(1 - ps)`. This creates a "pseudo-population" where treatment is independent of the covariates.

Compared to PSM, IPTW keeps every observation (better efficiency) but is more sensitive to extreme propensity scores (very small `ps` or `1 - ps` produces huge weights).

In [ ]:
# Weights (Horvitz-Thompson style).
sim['iptw'] = np.where(sim.treatment == 1, 1 / sim.ps, 1 / (1 - sim.ps))

# Trim extreme weights (standard practice: cap at 99th percentile) to control variance.
w_cap = sim['iptw'].quantile(0.99)
sim['iptw_trim'] = sim['iptw'].clip(upper=w_cap)

# Weighted outcome rates.
def weighted_mean(y, w):
    return np.average(y, weights=w)

p_t = weighted_mean(sim[sim.treatment == 1].outcome, sim[sim.treatment == 1].iptw_trim)
p_c = weighted_mean(sim[sim.treatment == 0].outcome, sim[sim.treatment == 0].iptw_trim)
iptw_rd = p_t - p_c

print(f'IPTW risk difference = {iptw_rd:+.3f}')
print(f'True treatment effect = {TRUE_TE:+.3f}')
print(f'Bias after IPTW = {iptw_rd - TRUE_TE:+.3f}')
print(f'\nMax weight before trim: {sim.iptw.max():.1f}')
print(f'Max weight after trim  : {sim.iptw_trim.max():.1f}  (99th pct)')

## 4. Balance diagnostics

Numeric estimates alone are not enough. You also need to check that the adjustment actually balanced the covariates. Two standard diagnostics:

- **Standardized mean difference (SMD)**: `(mean_treated - mean_control) / pooled_sd`. Rule of thumb: `|SMD| < 0.1` = well balanced.
- **Propensity score overlap plot**: the propensity distributions of treated and control should overlap. No overlap = no valid comparison in that region.

In [ ]:
def smd(x_t, x_c, w_t=None, w_c=None):
    """Weighted standardized mean difference."""
    if w_t is None:
        m_t, v_t = x_t.mean(), x_t.var()
    else:
        m_t = np.average(x_t, weights=w_t)
        v_t = np.average((x_t - m_t) ** 2, weights=w_t)
    if w_c is None:
        m_c, v_c = x_c.mean(), x_c.var()
    else:
        m_c = np.average(x_c, weights=w_c)
        v_c = np.average((x_c - m_c) ** 2, weights=w_c)
    return (m_t - m_c) / np.sqrt((v_t + v_c) / 2)


rows = [
    ('Age', 'Before adjustment',
     smd(sim[sim.treatment == 1].age, sim[sim.treatment == 0].age)),
    ('Age', 'After PSM',
     smd(treated.age, control_matched.age)),
    ('Age', 'After IPTW',
     smd(sim[sim.treatment == 1].age, sim[sim.treatment == 0].age,
         sim[sim.treatment == 1].iptw_trim, sim[sim.treatment == 0].iptw_trim)),
]

balance_tbl = pd.DataFrame(rows, columns=['Covariate', 'Sample', 'SMD']).round(3)
print(balance_tbl.to_string(index=False))
print('\n|SMD| < 0.1 is the standard threshold for adequate balance.')

In [ ]:
# Propensity score distributions - do treated and control overlap?
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sim[sim.treatment == 1].ps, bins=40, alpha=0.55, label='Treated',
        color='steelblue', density=True)
ax.hist(sim[sim.treatment == 0].ps, bins=40, alpha=0.55, label='Control',
        color='darkorange', density=True)
ax.set_xlabel('Propensity score')
ax.set_ylabel('Density')
ax.set_title('Propensity score overlap between treated and control')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Apply to real data: NCCTG colon trial

The NCCTG colon dataset (Moertel et al. 1990) is technically a randomized trial, but we can still exercise the machinery honestly: fit the propensity model, check that randomization already achieved balance (SMDs should be near zero without any adjustment), and see how the adjusted vs. unadjusted mortality contrasts compare.

**Comparison:** `Obs` (observation only) vs. `Lev+5FU` (levamisole + fluorouracil, the arm that established adjuvant 5-FU as standard of care).

In [ ]:
URL = 'https://vincentarelbundock.github.io/Rdatasets/csv/survival/colon.csv'
raw = pd.read_csv(URL, index_col=0)

# etype == 2: death event only (drop the recurrence rows).
colon = raw[raw['etype'] == 2].copy()

# Keep just the two-arm comparison; drop Lev alone.
colon = colon[colon['rx'].isin(['Obs', 'Lev+5FU'])].copy()
colon['treatment'] = (colon['rx'] == 'Lev+5FU').astype(int)
colon['event'] = colon['status']

# Covariates commonly used to characterize CRC prognosis at diagnosis.
covariates = ['age', 'sex', 'nodes', 'differ', 'extent', 'obstruct', 'perfor', 'adhere']
colon = colon.dropna(subset=covariates + ['treatment', 'event'])

print(f'N = {len(colon):,}  ({colon.treatment.sum():,} treated, {(colon.treatment==0).sum():,} control)')
print(f'Events: {colon.event.sum():,}  ({colon.event.mean()*100:.1f}%)')

In [ ]:
# 1. Naive contrast (no adjustment).
p_t = colon[colon.treatment == 1].event.mean()
p_c = colon[colon.treatment == 0].event.mean()
print(f'Naive mortality contrast (Lev+5FU vs Obs): {p_t - p_c:+.3f}')

# 2. Fit propensity model on the observed covariates.
X = colon[covariates].values
y = colon['treatment'].values
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X, y)
colon['ps'] = ps_model.predict_proba(X)[:, 1]

# 3. IPTW estimate.
colon['w'] = np.where(colon.treatment == 1, 1 / colon.ps, 1 / (1 - colon.ps))
colon['w_trim'] = colon['w'].clip(upper=colon['w'].quantile(0.99))
p_t_w = np.average(colon[colon.treatment == 1].event, weights=colon[colon.treatment == 1].w_trim)
p_c_w = np.average(colon[colon.treatment == 0].event, weights=colon[colon.treatment == 0].w_trim)
print(f'IPTW  mortality contrast (Lev+5FU vs Obs): {p_t_w - p_c_w:+.3f}')

# 4. SMDs before and after weighting (should already be small pre-adjustment if randomization worked).
rows = []
for cov in covariates:
    before = smd(colon.loc[colon.treatment == 1, cov],
                 colon.loc[colon.treatment == 0, cov])
    after  = smd(colon.loc[colon.treatment == 1, cov],
                 colon.loc[colon.treatment == 0, cov],
                 colon.loc[colon.treatment == 1, 'w_trim'],
                 colon.loc[colon.treatment == 0, 'w_trim'])
    rows.append((cov, before, after))

balance = pd.DataFrame(rows, columns=['Covariate', 'SMD before', 'SMD after']).round(3)
print('\nCovariate balance (SMD):')
print(balance.to_string(index=False))
print('\nExpectation: SMDs are near zero even BEFORE adjustment because NCCTG was randomized.')
print('IPTW would matter more on truly observational data (e.g., Medicare claims).')

## 6. Sensitivity to unmeasured confounding: the E-value

Every propensity method above assumes NO UNMEASURED CONFOUNDING - that you have adjusted for every variable that affects both treatment and outcome. That is untestable. What you *can* do is quantify how strong an unmeasured confounder would have to be to explain your finding away.

The **E-value** (VanderWeele & Ding 2017) is the minimum strength of association (on the risk-ratio scale) that an unmeasured confounder would need with BOTH treatment and outcome to fully account for the observed effect. Bigger E-value = more robust finding.

Formula: `E-value = RR + sqrt(RR * (RR - 1))`  where `RR >= 1` (invert for RR < 1).

In [ ]:
def evalue(rr):
    """E-value for a risk ratio (VanderWeele & Ding, Ann Intern Med 2017)."""
    if rr < 1:
        rr = 1 / rr
    return rr + np.sqrt(rr * (rr - 1))


# Convert IPTW risk difference to a risk ratio to feed into the E-value.
rr_iptw = p_t_w / p_c_w
ev = evalue(rr_iptw)
print(f'IPTW risk ratio (Lev+5FU vs Obs): {rr_iptw:.3f}')
print(f'E-value: {ev:.2f}')
print()
print(f'Interpretation: an unmeasured confounder would need to be associated with')
print(f'BOTH treatment and mortality by a risk ratio of at least {ev:.2f} to explain')
print(f'the observed protective effect away. Larger E-values mean stronger findings.')

## 7. When to use what

| Situation | Preferred method | Why |
|---|---|---|
| Well-overlapping propensity distributions, moderate N | PSM (1:1) | Easy to explain, matched sample is intuitive |
| Large N, some extreme propensities | IPTW with weight trimming | Uses all data, gains efficiency, trimming controls variance |
| Both, best of both | Doubly robust (AIPW / TMLE) | Consistent if EITHER the propensity model OR the outcome model is correct |
| Time-varying treatment or time-varying confounding | Marginal structural models with IPTW | Standard PSM breaks under time-varying confounding |
| Real-world sensitivity claim | Always report an E-value | Directly addresses the "what if there's an unmeasured confounder?" objection |

**Key libraries to explore next:**
- `causalinference` (Python): PSM + IPTW + doubly robust in one API.
- `dowhy` (Python): explicit causal DAG + method selection framework.
- `zEpid` (Python): epi-specific, IPTW + g-methods + marginal structural models.
- `sensmakr` (R): richer sensitivity analysis than E-value alone.

**What to try on the FAERS project:** FAERS itself cannot support propensity adjustment (spontaneous reports lack the covariate depth needed for a credible propensity model). But if you extend the SEER project (Phase 2 with the SEER DUA data) or add a Medicare claims analysis, the propensity framework is directly applicable.